In [ ]:
# General imports
import os
import pandas as pd
import numpy as np
from collections import Counter

# Input scaling imports
from sklearn.preprocessing import MinMaxScaler

# Regression & gene selection imports
import shap
from time import time
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import StratifiedKFold
from joblib import parallel_backend
from sklearn import metrics

# Wrapped functions definition

In [ ]:
def shap_f(feat_data, feat_y, fit_model, show=False):  
    """
    Implementing the SHAP function to be used after regression
    """
    shap.initjs()
    
    explainer = shap.TreeExplainer(fit_model)

    shap_values = explainer.shap_values(feat_data) 

    print(f'expected value = {explainer.expected_value}')
    print(f'mean = {np.mean(feat_y)}')


    # a shap value x each gene x each sample x each feature (in for loop the current feature)
    shap_df = pd.DataFrame(shap_values).rename(columns= dict(zip(list(range(0, len(feat_data.columns) + 1)), list(feat_data.columns))))
    
    # mean of the shap values assigned to the same genes in different samples
    shap_means = {}
    for col in list(shap_df.columns):
        shap_means[col] = np.mean(shap_df[col])
        
    shap_means_df = pd.DataFrame({"ref" : list(shap_means.keys()),  # gene_name
                                  "mean_shap_value": list(shap_means.values())})  
    
    # set show to True and uncomment the following rows and return part to obtain the worm-bees SHAP plot        
    # figure = plt.figure()
    
    # sort values to give a ranked list of shap values from positives to negatives
    return shap_means_df.sort_values(by="mean_shap_value", ascending=False) #, shap.summary_plot(shap_values, feat_data, max_display=40, show=show)

def minmax_hive_f(input_dir, data):
    
    scaler = MinMaxScaler()  # MinMax scaling [0,1]
    scaled_data = scaler.fit_transform(data)
    
    scaled_data_df = pd.DataFrame(scaled_data)
    scaled_data_df.set_index(data.index, inplace=True)  # reset index and columns for the scaled data
    scaled_data_df.columns = data.columns
    
    return scaled_data_df

def build_bins(n_bins, all_shaps, logic, silent=False):
    min_shaps = min(all_shaps)  # min of all the shap values x each gene x each feature in the latent space 
    max_shaps = max(all_shaps)  # max 

    
    if logic == "ar":
        step = round(((max_shaps - min_shaps)/n_bins), 4)
        bins = list(np.arange(min_shaps, max_shaps, step))
    else:
        bins, step = list(np.linspace(min_shaps, max_shaps, n_bins, endpoint=False, retstep=True))

    if not silent:
        print (f' - n_bins expected: {n_bins}\n' +
            f' - Actual number of bins: {len(bins)}\n' +
            f' - Intended bin interval size: {step}\n' +
            f' - Last Bin interval size: {max_shaps - bins[-1]}\n')
        
    return bins

def shap_bin_selection(bins, f_shaps, silent=False):
    """
    binning the shap distribution of each regression to automatically select important genes
    """    
    
    
    genes_in_bins = []  # how many genes falls in each bin

    for i in range(len(bins)-1):

        n_genes = len(list(f_shaps["mean_shap_value"][(f_shaps.mean_shap_value >= bins[i]) & (f_shaps.mean_shap_value < bins[i+1])]))
    
        genes_in_bins.append(n_genes)

    i_biggest = genes_in_bins.index(max(genes_in_bins))  # the biggest bin in terms of gene counts fairly contains genes of no interest

    # hive select only genes with shap values in bins smaller or greater than this bins
    ref_bin_neg = f_shaps["mean_shap_value"][(f_shaps.mean_shap_value > bins[i_biggest -1]) & (f_shaps.mean_shap_value <= bins[i_biggest])]
    ref_bin_pos = f_shaps["mean_shap_value"][(f_shaps.mean_shap_value > bins[i_biggest + 1]) & (f_shaps.mean_shap_value <= bins[i_biggest + 2])]

    mean_ref_bin_neg = ref_bin_neg.mean()
    mean_ref_bin_pos = ref_bin_pos.mean()
    
    if not silent: print(genes_in_bins) 

    return mean_ref_bin_neg, mean_ref_bin_pos, genes_in_bins

def build_meta(n_folds):
    """
    this will results in a final output files containing the performance measures of rfr per fold per latent features
    """
    
    folds = []
    measures = []
    for n in range(n_folds):
        folds.extend([str(n+1)]*5)
        measures.extend(["MSE", "RMSE", "MAE", "R2", "MAAE"])
    folds.extend(["-"]*4)
    measures.extend(["AvgMSE", "StdMSE", "AvgRMSE", "StdRMSE"])

    return {"Fold": folds, "Measure": measures}



# gHVIE Random Forest Regression

In [ ]:
## MODIFY THE FOLLOWING VECTOR ACCORDINGLY TO THE CLASS IN YOUR DATASET 
## PAY ATTENTION TO MAINTAIN THE ORDER OF THE SAMPLES WHEN ASSIGN A CLASS (== A NUMBER) TO THEM

# set I/O paths
hive_input_dir = "./Data/" # yHIVE_results 
hive_output_dir =  "./gHIVE_results/"
y_classes = list(pd.read_csv(f"{hive_input_dir}rfr_classes_definition.tsv", sep="\t")["class"])

# declare if needed MinMax scaling 
minmax_scaling = True

latent_space = pd.read_csv(f"{hive_input_dir}yHIVE_latent_space.tsv", sep="\t", index_col=0)
input_for_regression =  pd.read_csv(f"{hive_input_dir}yHIVE_minmax_scaled_input_data.tsv", sep="\t", index_col=0) # the input for yHIVE-VAE
input_for_regression = input_for_regression.transpose()


latent_space_feat = latent_space[list(latent_space.columns)[0:]]

# MinMax scale latent features to be in the same value-magnitude of the genes that will be regressed
scaled_latent_space = minmax_hive_f(hive_input_dir, latent_space_feat) 

rfr_output_dir = f"{hive_output_dir}SHAP_per_fold/"
if not os.path.exists(rfr_output_dir):
    os.makedirs(rfr_output_dir)

mse = {}
rmse = {}
mae = {}
r2 = {}


all_shaps = [] 

reg_X = input_for_regression  

smaller_class = np.min(list(Counter(y_classes).values()))

meta = build_meta(smaller_class)

skf = StratifiedKFold(n_splits=smaller_class, shuffle=True, random_state=42)

for f in range(len(list(scaled_latent_space.columns))):
    print("Latent Feature", f)
    i = 0
    reg_y= scaled_latent_space[str(f)].values
    start = time()
    meta[f"LF{f+1}"] = []

    best_rmse_index = None
    best_rmse_value = float('inf')

    for train_index, test_index in skf.split(reg_X, y_classes):
        print("fold", i)
        x_train = reg_X.iloc[train_index]
        x_test = reg_X.iloc[test_index]
        y_train = reg_y[train_index]
        y_test = reg_y[test_index]
        # print(train_index, '\t', test_index)

        # Random Forest Regression
        current_rfr = RandomForestRegressor(n_estimators=100, random_state=42) 
        
        with parallel_backend("threading", n_jobs=16):
            current_rfr.fit(x_train, y_train)

            ## Use the fitted model to predict the test data
            print("Predictions...")
            y_pred = current_rfr.predict(x_test)

        # RFR performances
        mse = metrics.mean_squared_error(y_test, y_pred)
        rmse = np.sqrt(mse)
        mae = metrics.mean_absolute_error(y_test, y_pred)
        r2 = metrics.r2_score(y_test, y_pred)
        maae = metrics.median_absolute_error(y_test, y_pred)
        
        ########### meta info files #########
        meta[f"LF{f+1}"] += [mse, rmse,mae,r2,maae]
        
        # uncomment the lines below to print per feature performances
        #print("Performances...")
        #print("MSE: ", mse)
        #print("RMSE: ", rmse)
    
        if rmse < best_rmse_value:
            best_rmse_value = rmse
            best_rmse_index = i
            best_rfr = current_rfr
        
        i += 1

    shap_ranked_mean = shap_f(x_test, reg_y, best_rfr) 

    print("SAVE...")
    shap_ranked_mean.to_csv(f"{rfr_output_dir}/rfr_SHAP_ranked_LF{f+1}.tsv", sep="\t", index=False) 
        

    meta[f"LF{f+1}"] += [np.mean([meta[f"LF{f+1}"][0], meta[f"LF{f+1}"][5], meta[f"LF{f+1}"][10]]),
                         np.std([meta[f"LF{f+1}"][0], meta[f"LF{f+1}"][5], meta[f"LF{f+1}"][10]]),
                         np.mean([meta[f"LF{f+1}"][1], meta[f"LF{f+1}"][6], meta[f"LF{f+1}"][11]]),
                         np.std([meta[f"LF{f+1}"][1], meta[f"LF{f+1}"][6], meta[f"LF{f+1}"][11]])]

    print("Overall LF time: ", (time() - start)/60, "minutes")


# gHIVE genes selection

In [ ]:
def bin_selection(n_bins):

    all_shaps = []
    for f in range(len(latent_space.columns)):
        file_shaps = pd.read_csv(f"{rfr_output_dir}/rfr_SHAP_ranked_LF{f+1}.tsv", sep="\t")
        shap_values = file_shaps.mean_shap_value
        all_shaps += list(shap_values)

    selected_shap_neg = {} # per feature gene shap values top negatives
    selected_shap_pos = {} # and positives

    bins = build_bins(n_bins, all_shaps, "lin")


    histo = pd.DataFrame(columns=[f'bin{i+1}' for i in range(len(bins)-1)])
    for f in range(len(latent_space.columns)):
        f_shaps = pd.read_csv(f"{rfr_output_dir}/rfr_SHAP_ranked_LF{f+1}.tsv",
                            sep="\t")
        
        neg, pos, distr = shap_bin_selection(bins, f_shaps, silent=True)  

        histo.loc[len(histo)] = distr
        
        selected_shap_neg[f] = list(f_shaps["ref"][f_shaps.mean_shap_value <= neg]) # all genes in the smaller bins below the neg shap mean
        selected_shap_pos[f] = list(f_shaps["ref"][f_shaps.mean_shap_value >= pos]) # all genes in the higher bins above the pos shap mean
        
    selected_genes = []
    for el in selected_shap_neg:
        genes = selected_shap_neg[el]
        selected_genes += genes

    for el in selected_shap_pos:
        genes = selected_shap_pos[el]
        selected_genes += genes  

    print(f'genes selected: {len(np.unique(selected_genes))}')

    # save the name of selected genes, the final output of HIVE
    with open(f"./HIVE_tomatoraw_ly3_ls80_ep300_gene_selection_138bins.txt", "w") as hive_out:
        for g in list(np.unique(selected_genes)):
            hive_out.write(f"{g}\n")
    

bin_selection(138) ##### change here the number of bins